# Day 3, session 3B: Regression workflow with XGBoost

**Estimated core time:** 60 minutes  
**Dataset:** `data/housing.csv`  
**Main repetition:** load → define preprocessing → cross-validate complete pipelines → fit → standard regression metrics  
**New fitting problem:** predicting a continuous outcome (`price_eur`)

All exercise code cells are empty. This notebook asks you to transfer the same reliable workflow from classification to regression.

## Why this matters

Classification and regression use the same experimental discipline but different targets, splitters, predictions, and metrics. A strong interview answer separates the stable workflow from the problem-specific choices. Housing prices also make metric units tangible: MAE and RMSE are in euros, while R-squared is unitless.

## Learning objectives

By the end, you should be able to:

1. Translate the end-to-end pipeline and cross-validation workflow from classification to regression.
2. Establish a mean-prediction baseline and compare it with linear, random-forest, and XGBoost regressors.
3. Calculate and interpret MAE, RMSE, and R-squared.
4. Create and interpret a residual plot with plotnine.
5. Diagnose classification code that was incorrectly copied into a regression workflow.

## R-to-Python bridge

| R modeling idea | Python / scikit-learn idea |
|---|---|
| continuous outcome in `lm()` | continuous `y` with a regressor |
| mean-only/null model | `DummyRegressor(strategy="mean")` |
| ordinary v-fold CV | `KFold` |
| RMSE / MAE / R-squared | scorer names and metric functions |
| fitted values | `model.predict(X)` |
| residuals | `observed - predicted` |

Scikit-learn's cross-validation scorers are maximized. Losses therefore appear as negative values such as `neg_mean_absolute_error`; negate them when reporting MAE or RMSE in their natural, positive units.

## What changes from classification?

```text
classification                    regression
binary/categorical y              continuous y
StratifiedKFold                   KFold
predict + predict_proba           predict
accuracy/precision/recall/F1/AUC  MAE/RMSE/R-squared
DummyClassifier                   DummyRegressor
XGBClassifier                     XGBRegressor
```

The split-before-preprocessing rule, pipeline structure, fold-only fitting, baseline comparison, and untouched test set do **not** change.

## Suggested pacing

- 0–12 min: load, audit, and split the housing data
- 12–25 min: reconstruct preprocessing and regression pipelines
- 25–40 min: cross-validation and model comparison
- 40–52 min: final metrics and residual plot
- 52–60 min: debugging, retrieval, and interview drill

## Task 1 — Load the housing data and identify the learning problem

Load `data/housing.csv` as `housing`. Report:

- shape and dtypes,
- missing values,
- summary statistics for `price_eur`,
- whether `property_id` is unique.

Create a plotnine histogram of `price_eur`. Describe its units, range, and skewness.

Before running code, predict the type and shape of `housing["price_eur"]` and `housing[["price_eur"]]`. Which one is the conventional shape for scikit-learn's `y`?

## Task 2 — Define predictors and reserve the test set

Create:

- `y` as `price_eur`,
- `X` without `price_eur` or `property_id`,
- numeric and categorical feature-name lists.

Treat boolean columns deliberately: include them with categorical features so they are explicitly encoded.

Reserve 20% for testing with `random_state=42`. Do **not** use `stratify=y`: the target is continuous. Confirm the four shapes and compare training/test target summaries.

Acceptance check: `X` should have 8 columns.

## Task 3 — Reconstruct preprocessing and four complete pipelines

Build the familiar `preprocessor`:

- numeric: median imputation and standardization;
- categorical/boolean: most-frequent imputation and one-hot encoding with `handle_unknown="ignore"` and `sparse_output=False`.

Then create four full pipelines, each with a fresh cloned preprocessor:

```python
DummyRegressor(strategy="mean")
LinearRegression()
RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
)
XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42,
)
```

Store them in a dictionary. Do not fit anything yet. Explain which estimator is the baseline and why a baseline is essential.

## Task 4 — Define regression cross-validation and metrics

Create one `KFold` with five folds, shuffling, and `random_state=42`.

Define this scoring dictionary:

```python
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}
```

Before fitting, predict:

1. Why will validation MAE and RMSE be negative in the raw output?
2. Which reported metrics should be smaller, and which should be larger?
3. Which metric is most sensitive to a few very large errors?

Then run `cross_validate` for one pipeline and inspect the keys and array shapes of its result.

## Task 5 — Cross-validate all four pipelines

Loop over the pipeline dictionary using only `X_train` and `y_train`. Build a summary with, for every model:

- mean and standard deviation of validation MAE in euros,
- mean and standard deviation of validation RMSE in euros,
- mean and standard deviation of validation R-squared.

Convert negative cross-validation losses back to positive MAE/RMSE before reporting. Sort by mean RMSE, ascending.

Acceptance criteria: all models use the exact same `cv` object and folds; no test data is used; each summary row is based on five validation scores.

## Task 6 — Interpret model quality and metric units

Answer with results from your CV table:

1. How many euros is the typical absolute error for each model?
2. How much does each trained model improve over predicting the training mean?
3. Why is RMSE larger than or equal to MAE?
4. Does the ranking change between MAE and RMSE?
5. What does the best R-squared say—and what does it **not** say?

Store a concise recommendation in a string called `model_recommendation` and display it.

## Task 7 — Fit the selected workflow and evaluate the test set once

Choose one pipeline based on cross-validation, fit it on all training rows, and predict `price_eur` for `X_test`.

Calculate:

- `mean_absolute_error`,
- `root_mean_squared_error`,
- `r2_score`.

Put the values in a one-row DataFrame with readable units. Compare test metrics with cross-validation means and discuss whether the difference looks compatible with fold-to-fold variation.

Acceptance check: predictions have shape `(number_of_test_rows,)`, contain no missing values, and are in euros.

## Task 8 — Build and read a residual plot

Create a DataFrame with:

- `observed_price`,
- `predicted_price`,
- `residual = observed_price - predicted_price`.

Before plotting, explain what a positive residual means under this definition.

With plotnine, plot predicted price on the x-axis and residual on the y-axis. Add a horizontal reference line at zero. A healthy plot should look roughly patternless around zero; describe any curvature, fan shape, or extreme errors you see.

Also report the five largest absolute residuals.

## Task 9 — Debug a classification workflow copied into regression

An AI assistant proposes:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    XGBClassifier(), X_train, y_train, cv=cv, scoring="accuracy"
)

model = XGBClassifier().fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]
```

Identify every classification-specific choice that is wrong for continuous house prices. Rewrite the estimator, splitter, scoring, prediction, and evaluation choices. Then identify the additional preprocessing problem that the snippet ignores.

## Retrieval practice — Task 10: rebuild the skeleton from memory

In a fresh set of variable names and without copying earlier cells, reconstruct this skeleton:

**CSV → X/y → train/test split → `ColumnTransformer` → `Pipeline` with `XGBRegressor` → `KFold` → cross-validated RMSE**

Use only one model and report the five positive fold RMSE values plus their mean and standard deviation.

Then point to every line that learns from data. Explain why none of those lines may see `X_test` or `y_test` yet.

## Interview questions

Answer aloud or add markdown cells:

1. Walk me through what stays the same and what changes when moving from binary classification to regression.
2. Compare MAE, RMSE, and R-squared. When might you prefer each?
3. Why should a regression model beat a mean baseline before you discuss sophisticated tuning?
4. What could go wrong if you compare CV RMSE from one model with test RMSE from another?

## Self-check and reflection

Record:

- Confidence from 1–5:
- Could you reconstruct preprocessing without looking back?
- Which classification habit did you have to change for regression?
- Can you explain MAE, RMSE, and R-squared in plain language?
- One thing you can now explain in an interview:
- What deserves another repetition?

## Optional extension — Where does the model fail? (outside the 60 minutes)

Attach `neighborhood` and `renovated` from the test rows to the residual DataFrame. Use pandas `groupby` to calculate count, MAE, mean residual, and RMSE by segment.

Identify segments with systematic underprediction or overprediction, while checking sample sizes. Explain why this error analysis is not the same as proving unfairness or causality.

As a second comparison, try modeling `log1p(price_eur)` and back-transform predictions with `expm1`. Compare errors in original euro units.

## Finish line

You have completed the core lesson when you can reproduce and explain:

**raw regression data → held-out test set → leakage-safe preprocessing → mean baseline → shared K-fold comparison → one fitted model → MAE/RMSE/R-squared → residual diagnosis**